In [18]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_absolute_error
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, GridSearchCV, TimeSeriesSplit
from sklearn.preprocessing import MinMaxScaler
from pathlib import Path
import sys
project_root = Path.cwd().parent
sys.path.append(str(project_root))

from src.fucntions import validate_agg_data, adf_test, Evaluator

In [43]:
California= pd.read_csv('../data/interim/Media_with_features.csv')

In [44]:
California[['year_month']].dtypes

year_month    str
dtype: object

In [45]:
California['log_rate'] = np.log1p(
    California['rate_per_100k']
)
adf_test(California['log_rate'], "VF Rate per 100k")

California['logdiff_rate']= California['log_rate'].diff()
California= California.dropna(subset= 'logdiff_rate')
adf_test(California['logdiff_rate'], 'VF logdiffed rates')

VF Rate per 100k
ADF Statistic : -0.3003
P-value       : 0.9255
Non-stationary
VF logdiffed rates
ADF Statistic : -8.1790
P-value       : 0.0000
Stationary


In [46]:
testcols= ['article_count','mean_title_sentiment','mean_title_risk','mean_body_risk','avg_sentiment','avg_risk','mean_body_sentiment', 'tweet_count','avg_likes']
artcols= ['article_count','mean_title_sentiment','mean_title_risk','mean_body_risk']
twcols= ['avg_sentiment','avg_risk','mean_body_sentiment','tweet_count','avg_likes']

In [47]:
#total is 166 months
onem= 165
threem= 163
sixm= 160
horizons=[1,3,6]

In [48]:
horizons = [1, 3, 6] 
splits = {} 
for horizon in horizons: 
     splits[horizon] = { 
	"X_train": X.iloc[:-horizon],
	"X_test": X.iloc[-horizon:], 
	"y_train": y.iloc[:-horizon], 
	"y_test": y.iloc[-horizon:] } 

X_train1=splits[1]['X_train'] 
X_train3=splits[3]['X_train'] 
X_train6=splits[6]['X_train'] 
Y_train1=splits[1]['y_train'] 
Y_train3=splits[3]['y_train'] 
Y_train6=splits[6]['y_train'] 
X_test1=splits[1]['X_test'] 
X_test3=splits[3]['X_test'] 
X_test6=splits[6]['X_test'] 
Y_test1=splits[1]['y_test'] 
Y_test3=splits[3]['y_test'] 
Y_test6=splits[6]['y_test']

In [49]:
X = California.drop(columns=[
    "year_month", "rate_per_100k", "logdiff_rate", "log_rate"
])

y = California["logdiff_rate"]

def walk_forward_splits(X, y, initial_train=120, horizon=1, step=1):
    splits = []

    for train_end in range(
        initial_train,
        len(X) - horizon + 1,
        step
    ):
        test_end = train_end + horizon

        splits.append({
            "X_train": X.iloc[:train_end],
            "y_train": y.iloc[:train_end],
            "X_test": X.iloc[train_end:test_end],
            "y_test": y.iloc[train_end:test_end]
        })

    return splits

splits = {
    horizon: walk_forward_splits(
        X, y, initial_train=120, horizon=horizon, step=1
    )
    for horizon in [1, 3, 6]
}

In [50]:
feature_sets = {
    "all_media": [
        'article_count',
        'mean_title_sentiment',
        'mean_title_risk',
        'mean_body_risk',
        'avg_sentiment',
        'avg_risk',
        'mean_body_sentiment',
        'tweet_count',
        'avg_likes'
    ],
    "articles": [
        'article_count',
        'mean_title_sentiment',
        'mean_title_risk',
        'mean_body_risk'
    ],
    "twitter": [
        'avg_sentiment',
        'avg_risk',
        'mean_body_sentiment',
        'tweet_count',
        'avg_likes'
    ]
}

In [51]:
##Use this one
def RandForest(Xtrain, Ytrain, Xtest, Ytest):

    # Make sure the target is one-dimensional
    Ytrain = np.asarray(Ytrain).reshape(-1)
    Ytest = np.asarray(Ytest).reshape(-1)

    fr = RandomForestRegressor(random_state=5)

    param_grid = {
        'n_estimators': [100, 200, 300],
        'max_depth': [None, 10, 20, 30],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4],
        'max_features': ['sqrt', 1.0]
    }

    tscv = TimeSeriesSplit(n_splits=5)

    grid_search = GridSearchCV(
        estimator=fr,
        param_grid=param_grid,
        cv=tscv,
        n_jobs=-1,
        scoring='neg_mean_squared_error',
        verbose=2
    )

    # Tune on training data only
    grid_search.fit(Xtrain, Ytrain)

    # Best estimator has already been fitted on Xtrain, Ytrain
    model = grid_search.best_estimator_

    # Forecast the held-out period
    fpreds = model.predict(Xtest)

    # Evaluate only against the corresponding test observations
    evaluation = Evaluator(Ytest, fpreds)

    return {
        'model': model,
        'evaluation': evaluation,
        'forecast': fpreds,
        'parameters': grid_search.best_params_,
        'cv_rmse': np.sqrt(-grid_search.best_score_)
    }

In [52]:
print(type(splits[1]))
print(splits[1].keys() if isinstance(splits[1], dict) else len(splits[1]))

<class 'list'>
46


In [53]:
param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", 1.0]
}

rf_results = {}

for feature_name, columns in feature_sets.items():

    # Tune only on the first 120 months
    first_split = splits[1][0]
    X_initial = first_split["X_train"][columns]
    y_initial = first_split["y_train"].to_numpy().ravel()

    search = GridSearchCV(
        RandomForestRegressor(random_state=5, n_jobs=1),
        param_grid=param_grid,
        cv=TimeSeriesSplit(n_splits=5),
        scoring="neg_mean_squared_error",
        n_jobs=-1,
        verbose=1
    )
    search.fit(X_initial, y_initial)
    best_params = search.best_params_

    for horizon in [1, 3, 6]:
        forecasts = []

        for i, split in enumerate(splits[horizon]):
            X_train = split["X_train"][columns]
            X_test = split["X_test"][columns]
            y_train = split["y_train"].to_numpy().ravel()
            y_test = split["y_test"].to_numpy().ravel()

            model = RandomForestRegressor(
                **best_params,
                random_state=5,
                n_jobs=-1
            )
            model.fit(X_train, y_train)
            predictions = model.predict(X_test)

            for actual, predicted in zip(y_test, predictions):
                forecasts.append({
                    "split": i,
                    "actual": actual,
                    "predicted": predicted
                })

        rf_results[(feature_name, horizon)] = pd.DataFrame(forecasts)

Fitting 5 folds for each of 162 candidates, totalling 810 fits
Fitting 5 folds for each of 162 candidates, totalling 810 fits
Fitting 5 folds for each of 162 candidates, totalling 810 fits


In [54]:

evaluation_results = {}

for (feature_name, horizon), df in rf_results.items():
    print(f"\nEvaluating: {feature_name} | {horizon}-month horizon")

    evaluation_results[(feature_name, horizon)] = Evaluator(
        df["actual"].to_numpy(),
        df["predicted"].to_numpy()
    )


Evaluating: all_media | 1-month horizon
Observations used: 46
MAE: 0.0740
RMSE: 0.0965
R²: -0.1166

Evaluating: all_media | 3-month horizon
Observations used: 132
MAE: 0.0755
RMSE: 0.0981
R²: -0.1220

Evaluating: all_media | 6-month horizon
Observations used: 246
MAE: 0.0751
RMSE: 0.0982
R²: -0.0925

Evaluating: articles | 1-month horizon
Observations used: 46
MAE: 0.0699
RMSE: 0.0951
R²: -0.0861

Evaluating: articles | 3-month horizon
Observations used: 132
MAE: 0.0713
RMSE: 0.0961
R²: -0.0763

Evaluating: articles | 6-month horizon
Observations used: 246
MAE: 0.0731
RMSE: 0.0974
R²: -0.0738

Evaluating: twitter | 1-month horizon
Observations used: 46
MAE: 0.0743
RMSE: 0.0963
R²: -0.1125

Evaluating: twitter | 3-month horizon
Observations used: 132
MAE: 0.0760
RMSE: 0.0979
R²: -0.1162

Evaluating: twitter | 6-month horizon
Observations used: 246
MAE: 0.0758
RMSE: 0.0983
R²: -0.0946


In [55]:
fmodel = RandomForestRegressor(n_estimators=300, max_depth=None, random_state=5, min_samples_split=10, min_samples_leaf=4, max_features='sqrt')
fmodel.fit(X_train1, Y_train1)
fpreds= fmodel.predict(X_test1)

Evaluator(California['logdiff_rate'],fpreds)

Observations used: 1
MAE: 0.0452
RMSE: 0.0452
R²: nan


c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\metrics\_regression.py:1295: UndefinedMetricWarning: R^2 score is not well-defined with less than two samples.
  warnings.warn(msg, UndefinedMetricWarning)


In [56]:
# Define parameter grid for tuning
param_grid = {
    'n_estimators': [100, 200, 300],       # Number of trees
    'max_depth': [None, 10, 20, 30],       # Tree depth
    'min_samples_split': [2, 5, 10],       # Min samples to split a node
    'min_samples_leaf': [1, 2, 4],         # Min samples at a leaf node
    'max_features': ['auto', 'sqrt']       # Features considered at each split
}

fr= RandomForestRegressor(random_state=5
                          )
# Grid search with 5-fold cross-validation
grid_search = GridSearchCV(
    estimator=fr,
    param_grid=param_grid,
    cv=5,
    n_jobs=-1,              # Use all CPU cores
    scoring='neg_mean_squared_error',
    verbose=2
)

# Fit the model
grid_search.fit(X_train1, Y_train1)

# Best parameters and score
print("Best Parameters:", grid_search.best_params_)
print("Best CV RMSE:", np.sqrt(-grid_search.best_score_))

# Evaluate on test set
best_rf = grid_search.best_estimator_
y_pred = best_rf.predict(X_test1)
test_rmse = np.sqrt(mean_squared_error(Y_test1, y_pred))
print("Test RMSE:", test_rmse)

Fitting 5 folds for each of 216 candidates, totalling 1080 fits


c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\model_selection\_validation.py:489: FitFailedWarning: 
540 fits failed out of a total of 1080.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
70 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\model_selection\_validation.py", line 851, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\base.py", line 1393, in wrapper
    estimator._validate_params()
    ~~~~~~~~~~~~~~~~~~~~~~~~~~^^
  File "c:

Best Parameters: {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 4, 'min_samples_split': 10, 'n_estimators': 300}
Best CV RMSE: 0.10111386087775673
Test RMSE: 0.0053804488991992545


In [57]:
fmodel = RandomForestRegressor(n_estimators=300, max_depth=None, random_state=5, min_samples_split=10, min_samples_leaf=4, max_features='sqrt')
fmodel.fit(X_train1, Y_train1)
fpreds= fmodel.predict(X_test1)

Evaluator(California['logdiff_rate'],fpreds)

# Define parameter grid for tuning
param_grid = {
    'n_estimators': [100, 200, 300],       # Number of trees
    'max_depth': [None, 10, 20, 30],       # Tree depth
    'min_samples_split': [2, 5, 10],       # Min samples to split a node
    'min_samples_leaf': [1, 2, 4],         # Min samples at a leaf node
    'max_features': ['auto', 'sqrt']       # Features considered at each split
}

fr= RandomForestRegressor(random_state=5
                          )
# Grid search with 5-fold cross-validation
grid_search = GridSearchCV(
    estimator=fr,
    param_grid=param_grid,
    cv=5,
    n_jobs=-1,              # Use all CPU cores
    scoring='neg_mean_squared_error',
    verbose=2
)

# Fit the model
grid_search.fit(X_train1, Y_train1)

# Best parameters and score
print("Best Parameters:", grid_search.best_params_)
print("Best CV RMSE:", np.sqrt(-grid_search.best_score_))

# Evaluate on test set
best_rf = grid_search.best_estimator_
y_pred = best_rf.predict(X_test1)
test_rmse = np.sqrt(mean_squared_error(Y_test1, y_pred))
print("Test RMSE:", test_rmse)

def RandFor(Xtrain, Ytrain, Xtest):
    grid= GridSearchCV(
        estimator=fr, 
        param_grid=param_grid,
        cv=5,
        n_jobs=-1,
        scoring='neg_mean_squared_error',
        verbose=2
    )

    grid_search.fit(Xtrain, Ytrain)

    parameters=grid_search.best_params_

    max_depth= parameters['max_depth']
    max_features=parameters['max_features']
    min_sample_leaf=parameters['min_samples_leaf']
    min_samples_split=parameters['min_samples_split']
    n_estimators=parameters['n_estimators']

    model= RandomForestRegressor(max_depth=max_depth, max_features=max_features, min_samples_leaf=min_sample_leaf, min_samples_split=min_samples_split, n_estimators=n_estimators)

    fmodel.fit(Xtrain, Ytrain)
    fpreds= fmodel.predict(Xtest)

    evaluation=Evaluator(California['logdiff_rate'], fpreds)

    return{
        'model':model,
        "evaluation": evaluation,
        'forecast':fpreds,
        'parameters': parameters
    }
        


c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\metrics\_regression.py:1295: UndefinedMetricWarning: R^2 score is not well-defined with less than two samples.
  warnings.warn(msg, UndefinedMetricWarning)


Observations used: 1
MAE: 0.0452
RMSE: 0.0452
R²: nan
Fitting 5 folds for each of 216 candidates, totalling 1080 fits


c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\model_selection\_validation.py:489: FitFailedWarning: 
540 fits failed out of a total of 1080.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
426 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\model_selection\_validation.py", line 851, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\base.py", line 1393, in wrapper
    estimator._validate_params()
    ~~~~~~~~~~~~~~~~~~~~~~~~~~^^
  File "c

Best Parameters: {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 4, 'min_samples_split': 10, 'n_estimators': 300}
Best CV RMSE: 0.10111386087775673
Test RMSE: 0.0053804488991992545


In [58]:
result1 = RandForest(
    X_train1, Y_train1, X_test1, Y_test1
)

result3 = RandForest(
    X_train3, Y_train3, X_test3, Y_test3
)

result6 = RandForest(
    X_train6, Y_train6, X_test6, Y_test6
)

Fitting 5 folds for each of 216 candidates, totalling 1080 fits


c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\metrics\_regression.py:1295: UndefinedMetricWarning: R^2 score is not well-defined with less than two samples.
  warnings.warn(msg, UndefinedMetricWarning)


Observations used: 1
MAE: 0.0082
RMSE: 0.0082
R²: nan
Fitting 5 folds for each of 216 candidates, totalling 1080 fits
Observations used: 3
MAE: 0.0560
RMSE: 0.0877
R²: -1.2775
Fitting 5 folds for each of 216 candidates, totalling 1080 fits
Observations used: 6
MAE: 0.0390
RMSE: 0.0633
R²: -1.2756


In [59]:
result1_twitter = RandForest(
    X_train1[twcols],
    Y_train1,
    X_test1[twcols],
    Y_test1
)

result1_twitter = RandForest(
    X_train3[twcols],
    Y_train3,
    X_test3[twcols],
    Y_test3
)

result6_twitter = RandForest(
    X_train6[twcols],
    Y_train6,
    X_test6[twcols],
    Y_test6
)

Fitting 5 folds for each of 216 candidates, totalling 1080 fits


c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\metrics\_regression.py:1295: UndefinedMetricWarning: R^2 score is not well-defined with less than two samples.
  warnings.warn(msg, UndefinedMetricWarning)


Observations used: 1
MAE: 0.0153
RMSE: 0.0153
R²: nan
Fitting 5 folds for each of 216 candidates, totalling 1080 fits
Observations used: 3
MAE: 0.0614
RMSE: 0.0940
R²: -1.6179
Fitting 5 folds for each of 216 candidates, totalling 1080 fits
Observations used: 6
MAE: 0.0509
RMSE: 0.0739
R²: -2.0969


In [60]:
result1_articles = RandForest(
    X_train1[artcols],
    Y_train1,
    X_test1[artcols],
    Y_test1
)

result3_articles = RandForest(
    X_train3[artcols],
    Y_train3,
    X_test3[artcols],
    Y_test3
)

result6_articles = RandForest(
    X_train6[artcols],
    Y_train6,
    X_test6[artcols],
    Y_test6
)

Fitting 5 folds for each of 216 candidates, totalling 1080 fits


c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\metrics\_regression.py:1295: UndefinedMetricWarning: R^2 score is not well-defined with less than two samples.
  warnings.warn(msg, UndefinedMetricWarning)


Observations used: 1
MAE: 0.0077
RMSE: 0.0077
R²: nan
Fitting 5 folds for each of 216 candidates, totalling 1080 fits
Observations used: 3
MAE: 0.0400
RMSE: 0.0576
R²: 0.0170
Fitting 5 folds for each of 216 candidates, totalling 1080 fits
Observations used: 6
MAE: 0.0340
RMSE: 0.0466
R²: -0.2311


In [5]:
Counties=pd.read_csv('../data/interim/Media_per_county.csv')

In [6]:
acols=['mean_title_positive', 'mean_title_neutral', 'mean_title_negative',
       'mean_title_sentiment', 'mean_body_positive', 'mean_body_neutral',
       'mean_body_negative', 'mean_title_risk', 'mean_body_sentiment',
       'mean_body_risk', 'article_count']
tcols=['min_sentiment', 'max_sentiment',
       'avg_sentiment', 'avg_positive', 'avg_negative', 'min_risk', 'max_risk',
       'avg_risk', 'tweet_count', 'avg_likes', 'total_likes']
mcols=['mean_title_positive', 'mean_title_neutral', 'mean_title_negative',
       'mean_title_sentiment', 'mean_body_positive', 'mean_body_neutral',
       'mean_body_negative', 'mean_title_risk', 'mean_body_sentiment',
       'mean_body_risk', 'article_count', 'min_sentiment', 'max_sentiment',
       'avg_sentiment', 'avg_positive', 'avg_negative', 'min_risk', 'max_risk',
       'avg_risk', 'tweet_count', 'avg_likes', 'total_likes']

In [7]:
feature_sets= {'all_media':mcols,
               'articles':acols,
               'twitter':tcols}

features=[mcols,acols,tcols]

target='rate_per_100k'

horizons= [1,3,6]

In [8]:
Counties = Counties.sort_values(
    ['County', 'year_month']
).copy()

county_data = {
    county: group.sort_values('year_month').reset_index(drop=True)
    for county, group in Counties.groupby('County')
}

In [9]:
county_model_data = {}

for county, df in county_data.items():
    df = df.sort_values('year_month').copy()

    df['log_rate'] = np.log1p(df[target])
    df['logdiff_rate'] = df['log_rate'].diff()

    df = df.dropna(subset=['logdiff_rate']).reset_index(drop=True)

    county_model_data[county] = {
        'df': df,
        'y': df['logdiff_rate'],
        'dates': df['year_month'],
        'X': {
            name: df[cols]
            for name, cols in feature_sets.items()
        }
    }

In [10]:
# Make Splits for every county
county_splits = {}

for county, data in county_model_data.items():
    county_splits[county] = {}

    y = data['y']

    for feature_name, X in data['X'].items():
        county_splits[county][feature_name] = {}

        for horizon in horizons:
            county_splits[county][feature_name][horizon] = {
                "X_train": X.iloc[:-horizon],
                "X_test": X.iloc[-horizon:],
                "y_train": y.iloc[:-horizon],
                "y_test": y.iloc[-horizon:]
            }

In [11]:
def walk_forward_splits(X, y, initial_train=120, horizon=1, step=1):
    splits = []

    for train_end in range(
        initial_train, len(X) - horizon + 1, step
    ):
        test_end = train_end + horizon

        splits.append({
            "X_train": X.iloc[:train_end],
            "X_test": X.iloc[train_end:test_end],
            "y_train": y.iloc[:train_end],
            "y_test": y.iloc[train_end:test_end]
        })

    return splits


horizons = [1, 3, 6]
initial_train = 120

county_splits = {}

for county, data in county_model_data.items():
    county_splits[county] = {}

    y = data['y']

    for feature_name, X in data['X'].items():
        county_splits[county][feature_name] = {}

        for horizon in horizons:
            county_splits[county][feature_name][horizon] = (
                walk_forward_splits(
                    X,
                    y,
                    initial_train=initial_train,
                    horizon=horizon,
                    step=1
                )
            )

In [12]:

forest_results = {}
forest_best_params = {}

param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", 1.0]
}

for county, feature_sets_for_county in county_splits.items():

    for feature_name, horizons_for_feature in feature_sets_for_county.items():

        # Get the first 120-month training window for this county and feature set
        initial_splits = horizons_for_feature[1]

        if not initial_splits:
            print(f"Skipping {county} / {feature_name}: insufficient data")
            continue

        first_split = initial_splits[0]
        X_initial = first_split["X_train"]
        y_initial = first_split["y_train"].to_numpy().ravel()

        # Tune hyperparameters once per county and feature set
        search = GridSearchCV(
            RandomForestRegressor(random_state=5, n_jobs=1),
            param_grid=param_grid,
            cv=TimeSeriesSplit(n_splits=5),
            scoring="neg_mean_squared_error",
            n_jobs=-1,
            verbose=0
        )

        search.fit(X_initial, y_initial)
        best_params = search.best_params_

        forest_best_params[(county, feature_name)] = best_params

        # Evaluate each horizon using walk-forward splits
        for horizon, splits_for_horizon in horizons_for_feature.items():

            forecasts = []

            for i, split in enumerate(splits_for_horizon):

                X_train = split["X_train"]
                X_test = split["X_test"]
                y_train = split["y_train"]
                y_test = split["y_test"]

                model = RandomForestRegressor(
                    **best_params,
                    random_state=5,
                    n_jobs=-1
                )

                model.fit(X_train, y_train)
                predictions = model.predict(X_test)

                for actual, predicted in zip(
                    y_test.to_numpy().ravel(), predictions
                ):
                    forecasts.append({
                        "split": i,
                        "actual": actual,
                        "predicted": predicted
                    })

            # Store all predictions for this county/feature set/horizon
            forest_results[(county, feature_name, horizon)] = pd.DataFrame(
                forecasts
            )

            print(
                f"{county} | {feature_name} | horizon={horizon} | "
                f"{len(splits_for_horizon)} splits complete"
            )


Alameda | all_media | horizon=1 | 35 splits complete
Alameda | all_media | horizon=3 | 33 splits complete
Alameda | all_media | horizon=6 | 30 splits complete


c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
c:\Users\z

Alameda | articles | horizon=1 | 35 splits complete


c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
c:\Users\z

Alameda | articles | horizon=3 | 33 splits complete
Alameda | articles | horizon=6 | 30 splits complete
Alameda | twitter | horizon=1 | 35 splits complete
Alameda | twitter | horizon=3 | 33 splits complete
Alameda | twitter | horizon=6 | 30 splits complete
Contra Costa | all_media | horizon=1 | 35 splits complete
Contra Costa | all_media | horizon=3 | 33 splits complete
Contra Costa | all_media | horizon=6 | 30 splits complete
Contra Costa | articles | horizon=1 | 35 splits complete
Contra Costa | articles | horizon=3 | 33 splits complete
Contra Costa | articles | horizon=6 | 30 splits complete
Contra Costa | twitter | horizon=1 | 35 splits complete
Contra Costa | twitter | horizon=3 | 33 splits complete
Contra Costa | twitter | horizon=6 | 30 splits complete
Fresno | all_media | horizon=1 | 35 splits complete
Fresno | all_media | horizon=3 | 33 splits complete
Fresno | all_media | horizon=6 | 30 splits complete
Fresno | articles | horizon=1 | 35 splits complete
Fresno | articles | hor

c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
c:\Users\zions\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
c:\Users\z

Merced | articles | horizon=1 | 35 splits complete
Merced | articles | horizon=3 | 33 splits complete
Merced | articles | horizon=6 | 30 splits complete
Merced | twitter | horizon=1 | 35 splits complete
Merced | twitter | horizon=3 | 33 splits complete
Merced | twitter | horizon=6 | 30 splits complete
Monterey | all_media | horizon=1 | 35 splits complete
Monterey | all_media | horizon=3 | 33 splits complete
Monterey | all_media | horizon=6 | 30 splits complete
Monterey | articles | horizon=1 | 35 splits complete
Monterey | articles | horizon=3 | 33 splits complete
Monterey | articles | horizon=6 | 30 splits complete
Monterey | twitter | horizon=1 | 35 splits complete
Monterey | twitter | horizon=3 | 33 splits complete
Monterey | twitter | horizon=6 | 30 splits complete
Orange | all_media | horizon=1 | 35 splits complete
Orange | all_media | horizon=3 | 33 splits complete
Orange | all_media | horizon=6 | 30 splits complete
Orange | articles | horizon=1 | 35 splits complete
Orange | arti

In [13]:
evaluation_rows = []
for (county, feature_name, horizon), df in forest_results.items():
    actual = df["actual"].to_numpy()
    predicted = df["predicted"].to_numpy()

    evaluation_rows.append({
        "County": county,
        "Model": "RandForest",
        "Feature Set": feature_name,
        "Horizon": horizon,
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": np.sqrt(mean_squared_error(actual, predicted)),
        "R2": (
            r2_score(actual, predicted)
            if len(actual) >= 2 else np.nan
        ),
        "N Predictions": len(actual)
    })

forest_evaluation_table = pd.DataFrame(evaluation_rows)

forest_evaluation_table = forest_evaluation_table.sort_values(
    ["County", "Horizon", "Feature Set"]
).reset_index(drop=True)

print(forest_evaluation_table)

      County       Model Feature Set  Horizon       MAE      RMSE        R2  \
0    Alameda  RandForest   all_media        1  0.167706  0.206943 -0.038548   
1    Alameda  RandForest    articles        1  0.163292  0.203356 -0.002855   
2    Alameda  RandForest     twitter        1  0.166446  0.213604 -0.106474   
3    Alameda  RandForest   all_media        3  0.164674  0.207240 -0.021831   
4    Alameda  RandForest    articles        3  0.161914  0.203147  0.018136   
..       ...         ...         ...      ...       ...       ...       ...   
184  Ventura  RandForest    articles        3  0.223026  0.262023 -0.125872   
185  Ventura  RandForest     twitter        3  0.208290  0.254371 -0.061076   
186  Ventura  RandForest   all_media        6  0.215169  0.252924 -0.062679   
187  Ventura  RandForest    articles        6  0.217744  0.257930 -0.105162   
188  Ventura  RandForest     twitter        6  0.204433  0.251026 -0.046792   

     N Predictions  
0               35  
1        

In [19]:
forest_evaluation_table.to_csv('RandForest_Media_Model_Evals.csv', index=False)

In [15]:
best_rmse_by_county_horizon = forest_evaluation_table.loc[
    forest_evaluation_table.groupby(['County', 'Horizon'])['RMSE'].idxmin()
].sort_values(['County', 'Horizon']).reset_index(drop=True)

best_rmse_by_county_horizon

,County,Model,Feature Set,Horizon,MAE,RMSE,R2,N Predictions
0,Alameda,RandForest,articles,1,0.163292,0.203356,-0.002855,35
1,Alameda,RandForest,articles,3,0.161914,0.203147,0.018136,99
2,Alameda,RandForest,articles,6,0.158230,0.199214,0.010585,180
3,Contra Costa,RandForest,articles,1,0.171592,0.221725,-0.014342,35
4,Contra Costa,RandForest,articles,3,0.172304,0.223266,-0.011252,99
...,...,...,...,...,...,...,...,...
58,Tulare,RandForest,articles,3,0.208868,0.277552,-0.042572,99
59,Tulare,RandForest,articles,6,0.215873,0.284371,-0.056601,180
60,Ventura,RandForest,all_media,1,0.214756,0.251151,-0.064397,35
61,Ventura,RandForest,twitter,3,0.208290,0.254371,-0.061076,99


In [16]:
best_mae_by_county_horizon = forest_evaluation_table.loc[
    forest_evaluation_table.groupby(['County', 'Horizon'])['MAE'].idxmin()
].sort_values(['MAE', 'County']).reset_index(drop=True)

best_mae_by_county_horizon

,County,Model,Feature Set,Horizon,MAE,RMSE,R2,N Predictions
0,Los Angeles,RandForest,twitter,6,0.065273,0.080578,-0.129160,180
1,Los Angeles,RandForest,twitter,3,0.065295,0.080688,-0.132217,99
2,Los Angeles,RandForest,twitter,1,0.067887,0.083734,-0.137428,35
3,Orange,RandForest,all_media,6,0.101119,0.136548,-0.106193,180
4,Orange,RandForest,all_media,3,0.106148,0.140752,-0.098436,99
...,...,...,...,...,...,...,...,...
58,Monterey,RandForest,twitter,3,0.347141,0.439271,-0.168346,99
59,Monterey,RandForest,twitter,6,0.350781,0.438183,-0.168045,180
60,Merced,RandForest,twitter,1,0.387464,0.542352,-0.130550,35
61,Merced,RandForest,twitter,3,0.397885,0.552696,-0.127579,99


In [17]:
r2_valid = forest_evaluation_table.dropna(subset=['R2'])

best_R2_by_county_horizon = r2_valid.loc[
    r2_valid.groupby(['County', 'Horizon'])['R2'].idxmax()
].sort_values(['R2', 'Horizon']).reset_index(drop=True)

best_R2_by_county_horizon

,County,Model,Feature Set,Horizon,MAE,RMSE,R2,N Predictions
0,San Luis Obispo,RandForest,articles,6,0.278707,0.360796,-0.193884,180
1,San Luis Obispo,RandForest,articles,3,0.278182,0.358801,-0.181413,99
2,Monterey,RandForest,articles,3,0.350414,0.438522,-0.164361,99
3,Fresno,RandForest,articles,1,0.133005,0.165400,-0.156619,35
4,Monterey,RandForest,twitter,1,0.341188,0.434076,-0.153965,35
...,...,...,...,...,...,...,...,...
58,San Diego,RandForest,all_media,1,0.114715,0.139131,0.031550,35
59,San Diego,RandForest,all_media,3,0.114280,0.139614,0.032239,99
60,San Diego,RandForest,all_media,6,0.116684,0.141654,0.032363,180
61,San Mateo,RandForest,articles,3,0.193695,0.256808,0.035896,99
